In [1]:
print(123)

123


In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

KeyboardInterrupt: 

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp'
)

In [ ]:
rec = ground_truth[0]
question = rec["question"]
question

'Can I still join the course if I just found out about it?'

In [ ]:
original_answer=doc_idx[rec['document']]['answer']

In [ ]:
answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [ ]:
print('original answer:', original_answer)
print('llm answer:', answer_llm)

original answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
llm answer: Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, ground_truth, generate_rag_answer)

# answers = []

# for answer_record in results:
#     answers.append(answer_record)

# df_answers = pd.DataFrame(answers)
# df_answers.to_csv("data/rag-answers-new.csv", index=False)

  0%|          | 0/565 [00:00<?, ?it/s]

# LLM as a judge

In [5]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [7]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [8]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [10]:
rec = answers[0]

In [11]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [12]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says late joining is allowed and that certificate eligibility requires submitting the project while submissions are still accepted. No key information is missing or incorrect.', score='good')

In [13]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.000234,
 'total_cost': 0.00045675}

In [14]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [15]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late enrollment is allowed, but certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [16]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [17]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/395 [00:00<?, ?it/s]

In [18]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [19]:
df_eval = pd.DataFrame(evaluations)

In [20]:
calc_total_price(usages)

0.25281599999999993

In [21]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 375/395 = 94.94%


In [22]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not address the question or...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer states the opposite of the groun...
38,How will I know when a module is actually read...,96286b4be4,bad,The ground truth explains that modules will ke...
72,I added billing but still get an insufficient_...,152af39a53,bad,The AI answer captures one key point from the ...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The AI answer fails to answer the question and...
